# Google Drive Model Playground

This notebook imports the project modules from a Google Drive copy of the repo. It then registers a drop-in model variant, optionally runs a short experiment, saves weights into the existing `checkpoints/<model_type>_<log_id>/` artifact layout, and exposes the two harnesses without connecting them to trainable models yet.

## Mount Google Drive And Import From Drive

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

DEFAULT_DRIVE_REPO = Path("/content/drive/MyDrive/multi-brain-quadruped-sim")
DRIVE_REPO = Path(os.environ.get("GOOGLE_DRIVE_REPO", str(DEFAULT_DRIVE_REPO))).expanduser()

if not (DRIVE_REPO / "train_headless.py").exists():
    search_root = Path("/content/drive/MyDrive")
    candidates = list(search_root.glob("**/multi-brain-quadruped-sim/train_headless.py")) if search_root.exists() else []
    if candidates:
        DRIVE_REPO = candidates[0].parents[0]

drive_markers = {"drive", "mydrive", "my drive", "google drive", "googledrive"}
path_text = str(DRIVE_REPO).lower().replace("_", " ")
if not any(marker in path_text for marker in drive_markers):
    raise RuntimeError("This notebook must import the repo from Google Drive. Set GOOGLE_DRIVE_REPO to your Drive clone path.")
if not (DRIVE_REPO / "train_headless.py").exists():
    raise FileNotFoundError(f"Could not find a repo checkout at {DRIVE_REPO}. Set GOOGLE_DRIVE_REPO first.")

sys.path = [entry for entry in sys.path if "multi-brain-quadruped-sim" not in entry]
sys.path.insert(0, str(DRIVE_REPO))

DRIVE_REPO

## Optional Dependency Install

Enable this in a fresh Colab runtime if imports fail because JAX, MuJoCo, FastAPI, or YAML dependencies are missing.

In [ ]:
INSTALL_REPO_REQUIREMENTS = False

if INSTALL_REPO_REQUIREMENTS:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(DRIVE_REPO / "requirements.txt")])

## Import Repo Modules From Google Drive

In [ ]:
from dataclasses import replace

from brains.config import load_runtime_spec
from brains.harnesses import DirectionHarness, HeadCameraHarness
from brains.jax_trainer import ESTrainer, PARAM_COUNT, apply_runtime_spec
from brains.models import ModelDefinition, list_model_definitions, register_model_definition
from brains.runtime import create_model_run_paths, discover_model_artifacts, write_model_manifest

import brains

module_path = Path(brains.__file__).resolve()
if str(DRIVE_REPO.resolve()) not in str(module_path):
    raise RuntimeError(f"Imported brains from {module_path}, expected it under {DRIVE_REPO}")

{"imported_from": str(module_path), "param_count": PARAM_COUNT}

## Register A Drop-In Model Variant

Until a second trainer implementation exists, notebook-created models should keep the current shared-trunk ES shape and parameter count.

In [ ]:
MODEL_TYPE = "notebook_shared_trunk_v1"
LOG_ID = "notebook_demo_001"
SEED = 7

definition = ModelDefinition(
    type=MODEL_TYPE,
    architecture="shared_trunk_motor_lanes",
    trainer="openai_es",
    input_size=48,
    output_size=4,
    parameter_count=PARAM_COUNT,
    description="Google Drive notebook variant of the current shared trunk ES trainer.",
)
register_model_definition(definition, persist=True)

spec = load_runtime_spec(DRIVE_REPO / "configs" / "smoke.yaml")
spec = replace(
    spec,
    name="notebook-smoke",
    model=replace(
        spec.model,
        type=MODEL_TYPE,
        architecture=definition.architecture,
        trainer=definition.trainer,
        description=definition.description,
    ),
)
apply_runtime_spec(spec)

[model.to_dict() for model in list_model_definitions()]

## Create Or Train Weights

Keep `RUN_GENERATIONS = 0` for a fast random-policy artifact. Increase it for local experiments. Use PACE for real runs.

In [ ]:
RUN_GENERATIONS = 0

trainer = ESTrainer(seed=SEED, spec=spec, model_id=f"{MODEL_TYPE}_{LOG_ID}", log_id=LOG_ID)
for _ in range(RUN_GENERATIONS):
    trainer.run_generation()

{
    "model_type": spec.model.type,
    "generation": trainer.state.generation,
    "mean_reward": trainer.state.mean_reward,
    "best_reward": trainer.state.best_reward,
    "param_count": trainer.param_count,
}

## Save Into The Existing Model Path Layout

This writes `latest.npz` and `manifest.json` under the Google Drive repo's `checkpoints/` directory. The current viewer discovers these artifacts.

In [ ]:
paths = create_model_run_paths(DRIVE_REPO / "checkpoints", MODEL_TYPE, LOG_ID)
trainer.model_id = paths.id
trainer.log_id = paths.log_id
saved_path = trainer.save_checkpoint(paths.latest_path)
write_model_manifest(
    paths,
    spec,
    saved_path,
    generation=trainer.state.generation,
    best_reward=trainer.state.best_reward,
    mean_reward=trainer.state.mean_reward,
)

{
    "model_id": paths.id,
    "latest": str(paths.latest_path),
    "manifest": str(paths.manifest_path),
    "visible_to_viewer": [artifact.id for artifact in discover_model_artifacts(DRIVE_REPO / "checkpoints")],
}

## Harness 1: Direction Options

This is not connected to models yet. It maps language-like directions to available scripted options and motor targets.

In [ ]:
direction_harness = DirectionHarness(spec)
plan = direction_harness.compile("trot forward then turn left")

{
    "available_options": [option.name for option in direction_harness.available_options()],
    "plan": [command.option for command in plan.commands],
    "first_target_velocity": direction_harness.target_velocity(plan.commands[0].option, 0.25).tolist(),
}

## Harness 2: Head Camera For VLA Work

This builds on the direction harness and adds a front-mounted camera contract named `head_camera`.

In [ ]:
camera_harness = HeadCameraHarness(spec)
camera_xml = camera_harness.build_camera_xml()

{
    "camera_name": camera_harness.camera.name,
    "image_shape": [camera_harness.camera.height, camera_harness.camera.width, 3],
    "camera_xml_contains_head_camera": "head_camera" in camera_xml,
}

## PACE Command For The Same Model Type

After `configs/model_registry.json` exists in the Drive-backed repo copy, use the same `MODEL_TYPE` and `LOG_ID` on PACE.

In [ ]:
print(f"CONFIG=configs/smoke.yaml MODEL_TYPE={MODEL_TYPE} LOG_ID={LOG_ID} GENERATIONS=50 SEED={SEED} sbatch train.sbatch")